# 注解 Target

In [1]:
import set_env

In [2]:
import os
import sys
import numpy as np

import tvm
import tvm.relay.testing
import tvm.relay.transform as transform
from tvm import relay
from tvm import runtime
from tvm.contrib import utils

## 注解多 end

In [ ]:
@tvm.ir.register_op_attr("nn.relu", "target.test")
def relu(expr):  # pylint: disable=unused-variable
    # 注册 nn.relu 算子在 test 目标上的处理函数
    return True

In [4]:
def before():
    x = relay.var("x", shape=(10, 10))
    r = relay.nn.relu(x)
    a_1 = relay.abs(r)
    a_2 = relay.abs(r)
    out = relay.add(a_1, a_2)
    f = relay.Function([x], out)
    mod = tvm.IRModule.from_expr(f)
    return mod

`relu` 在 `test` 目标上执行，其余算子在 `default` 目标上执行：

In [7]:
# 测试两种场景：不注解非调用操作和注解非调用操作
for annotate_non_call_ops in [False, True]:
    # 应用 AnnotateTarget 转换
    result = transform.AnnotateTarget("test", annotate_non_call_ops)(before())
    if annotate_non_call_ops:
        result.show()

## 类型传播

In [9]:
target = "test_type_propagation"

@tvm.ir.register_op_attr("nn.relu", "target." + target)
def relu(expr):  # pylint: disable=unused-variable
    return expr.args[0].checked_type.dtype == "float32"

def before():
    x = relay.var("x", shape=(10, 10))
    r = relay.nn.relu(x)
    out = relay.nn.relu(r)
    f = relay.Function([x], out)
    mod = tvm.IRModule.from_expr(f)
    return mod

for annotate_non_call_ops in [False, True]:
    # If the type isn't propogated, then the relu checker function will fail to get the dtype.
    assert transform.AnnotateTarget(target, annotate_non_call_ops)(before())

In [10]:
transform.AnnotateTarget(target, True)(before()).show()

## RefCreate

In [11]:
target = "relu"

@tvm.ir.register_op_attr("nn.relu", "target." + target)
def annotate(expr):
    return True

def before():
    ref = relay.expr.RefCreate(relay.const(1.0))
    r = relay.expr.RefWrite(ref, relay.nn.relu(relay.expr.RefRead(ref)))
    return tvm.IRModule.from_expr(r)

In [12]:
result = transform.AnnotateTarget(target, annotate_non_call_ops)(before())
result.show()

## 测试 TupleNode 在 AnnotateTarget 转换中的处理

In [ ]:
# 定义目标编译器标识
target = "test_tuple"

# 注册 nn.relu 操作在指定目标上的处理函数
@tvm.ir.register_op_attr("nn.relu", "target." + target)
def relu(expr):  # pylint: disable=unused-variable
    return True

# 注册 concatenate 操作在指定目标上的处理函数
@tvm.ir.register_op_attr("concatenate", "target." + target)
def concatenate(expr):  # pylint: disable=unused-variable
    return True

 验证当元组被支持的节点包围时，元组是否被包含在注解

In [14]:
def before():
    x = relay.var("x", shape=(10, 5))
    y = relay.var("y", shape=(10, 5))
    a_1 = relay.nn.relu(x)
    a_2 = relay.nn.relu(y)
    out = relay.concatenate((a_1, a_2), axis=1)
    f = relay.Function([x, y], out)
    mod = tvm.IRModule.from_expr(f)
    return mod
result = transform.AnnotateTarget(target, annotate_non_call_ops)(before())
result.show()

## 测试复合函数

In [16]:
def before():
    a = relay.var("a", shape=(10, 10))
    b = relay.var("b", shape=(10, 10))

    # add_relu function
    in_1 = relay.var("in_1", shape=(10, 10))
    in_2 = relay.var("in_2", shape=(10, 10))
    add_node = relay.add(in_1, in_2)
    relu_node = relay.nn.relu(add_node)
    add_relu = relay.Function([in_1, in_2], relu_node)
    add_relu = add_relu.with_attr("Composite", "test.add_relu")

    # merged function
    r = relay.Call(add_relu, [a, b])
    f = relay.Function([a, b], r)
    mod = tvm.IRModule.from_expr(f)
    return mod
result = transform.AnnotateTarget(target, annotate_non_call_ops)(before())
result.show()

## 测试双目标

In [17]:
@tvm.ir.register_op_attr("nn.relu", "target.double.A")
def relu(expr):  # pylint: disable=unused-variable
    return True

def before():
    x = relay.var("x", shape=(10, 5))
    a_1 = relay.nn.relu(x)
    mod = tvm.IRModule.from_expr(a_1)
    return mod

for annotate_non_call_ops in [True, False]:
    mod = before()
    mod1 = transform.AnnotateTarget("double.A", annotate_non_call_ops)(mod)
    mod2 = transform.AnnotateTarget("double.A", annotate_non_call_ops)(mod1)
    tvm.ir.assert_structural_equal(mod1, mod2)

## 测试不同目标

In [18]:
@tvm.ir.register_op_attr("nn.relu", "target.different.A")
def relu(expr):  # pylint: disable=unused-variable
    return True

@tvm.ir.register_op_attr("add", "target.different.B")
def relu(expr):  # pylint: disable=unused-variable
    return True

def before():
    x = relay.var("x", shape=(10, 5))
    a_1 = relay.nn.relu(x)
    b_1 = relay.add(a_1, a_1)
    mod = tvm.IRModule.from_expr(b_1)
    return mod

for annotate_non_call_ops in [True, False]:
    mod = before()
    mod1 = transform.AnnotateTarget("different.A", annotate_non_call_ops)(mod)
    mod1 = transform.AnnotateTarget("different.B", annotate_non_call_ops)(mod1)
    mod2 = transform.AnnotateTarget(["different.A", "different.B"], annotate_non_call_ops)(mod)
    tvm.ir.assert_structural_equal(mod1, mod2)

In [19]:
@tvm.ir.register_op_attr("nn.relu", "target.A")
def relu(expr):  # pylint: disable=unused-variable
    return True

@tvm.ir.register_op_attr("add", "target.B")
def add(expr):  # pylint: disable=unused-variable
    return True

def before():
    x = relay.var("x", shape=(10, 5))
    a_1 = relay.nn.relu(x)
    a_2 = relay.abs(a_1)
    a_3 = relay.nn.relu(a_1)
    out = relay.add(a_2, a_3)

    f = relay.Function([x], out)
    mod = tvm.IRModule.from_expr(f)
    return mod

for annotate_non_call_ops in [True, False]:
    mod = transform.AnnotateTarget("A", annotate_non_call_ops)(before())
    mod = transform.AnnotateTarget("B", annotate_non_call_ops)(mod)
    expected = transform.AnnotateTarget(["A", "B"], annotate_non_call_ops)(before())
    tvm.ir.assert_structural_equal(expected, mod)

## 其他

In [21]:
def test_ends_with_tuple():
    """测试以元组结尾的计算图在AnnotateTarget转换中的处理

    此测试用例验证当计算图以元组（Tuple）结尾，并且包含TupleGetItem操作时，
    AnnotateTarget转换能否正确地处理这些结构并添加适当的编译器注解。
    """
    # 定义目标编译器标识
    trgt = "clip"

    # 注册clip操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("clip", "target." + trgt)
    def relu(expr):  # pylint: disable=unused-variable
        return True

    def get_model(get_item):
        """创建测试模型

        参数:
            get_item: 布尔值，表示是否使用TupleGetItem操作获取元组中的元素

        返回:
            包含clip操作和元组结构的IRModule
        """
        # 创建输入变量
        a = relay.var("a", shape=(1, 16, 16, 4), dtype="uint8")
        # 应用clip操作
        z = relay.op.clip(a, 0, 255)
        # 对同一输入应用不同范围的clip操作
        b = relay.op.clip(z, 0, 15)
        c = relay.op.clip(z, 16, 31)
        # 创建包含两个clip结果的元组
        t = relay.Tuple((c, b))
        # 根据参数决定是否使用TupleGetItem获取元组中的第二个元素
        tgi = relay.TupleGetItem(t, 1) if get_item else t
        foo = relay.Function([a], tgi)
        # 返回创建的IR模块
        return tvm.IRModule.from_expr(tgi)

    def get_expected(annotate_non_call_ops, get_item):
        """生成预期的注解后模型

        参数:
            annotate_non_call_ops: 布尔值，表示是否为非调用操作添加注解
            get_item: 布尔值，表示是否使用TupleGetItem操作

        返回:
            预期的包含正确编译器注解的IRModule
        """
        # 创建输入变量
        a_ = relay.var("a", shape=(1, 16, 16, 4), dtype="uint8")
        # 为输入添加编译器开始注解
        a = relay.annotation.compiler_begin(a_, trgt)
        # 应用clip操作
        z = relay.op.clip(a, 0, 255)
        # 为第一个clip结果添加编译器结束注解
        z1 = relay.annotation.compiler_end(z, trgt)
        # 为z1添加编译器开始注解
        z1 = relay.annotation.compiler_begin(z1, trgt)
        # 应用第二个clip操作
        b = relay.op.clip(z1, 0, 15)
        # 为b添加编译器结束注解
        b = relay.annotation.compiler_end(b, trgt)
        # 根据参数决定是否为b添加编译器开始注解
        b = relay.annotation.compiler_begin(b, trgt) if annotate_non_call_ops else b
        # 为z添加编译器结束注解
        z2 = relay.annotation.compiler_end(z, trgt)
        # 为z2添加编译器开始注解
        z2 = relay.annotation.compiler_begin(z2, trgt)
        # 应用第三个clip操作
        c = relay.op.clip(z2, 16, 31)
        # 为c添加编译器结束注解
        c = relay.annotation.compiler_end(c, trgt)
        # 根据参数决定是否为c添加编译器开始注解
        c = relay.annotation.compiler_begin(c, trgt) if annotate_non_call_ops else c
        # 创建包含c和b的元组
        t = relay.Tuple((c, b))
        # 根据参数决定是否为元组添加编译器结束注解
        t = relay.annotation.compiler_end(t, trgt) if annotate_non_call_ops else t
        if get_item:
            # 如果使用TupleGetItem，根据参数决定是否为元组添加编译器开始注解
            t = relay.annotation.compiler_begin(t, trgt) if annotate_non_call_ops else t
            # 获取元组中的第二个元素
            tgi = relay.TupleGetItem(t, 1)
            # 根据参数决定是否为TupleGetItem结果添加编译器结束注解
            tgi = relay.annotation.compiler_end(tgi, trgt) if annotate_non_call_ops else tgi
        else:
            tgi = t
        foo = relay.Function([a_], tgi)
        # 返回预期的IR模块
        return tvm.IRModule.from_expr(foo)

    # 测试所有组合场景
    for get_item in [True, False]:
        for annotate_non_call_ops in [False, True]:
            # 获取测试模型
            mod = get_model(get_item)
            # 应用AnnotateTarget转换
            mod = transform.AnnotateTarget("clip", annotate_non_call_ops)(mod)
            # 获取预期结果并应用类型推断
            expected = transform.InferType()(get_expected(annotate_non_call_ops, get_item))
            # 验证实际结果与预期结果是否结构相等
            tvm.ir.assert_structural_equal(expected, mod)


def test_if_else():
    """测试条件分支（If-else）节点在AnnotateTarget转换中的处理

    此测试用例验证当计算图包含条件分支结构，并且被支持的操作节点包围时，
    AnnotateTarget转换能否正确地为这些结构添加编译器注解。
    """
    # 定义目标编译器标识
    target = "test_if_else"

    # 注册equal操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("equal", "target." + target)
    def relu(expr):  # pylint: disable=unused-variable
        return True

    # 注册tanh操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("tanh", "target." + target)
    def tanh(expr):  # pylint: disable=unused-variable
        return True

    # 注册sigmoid操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("sigmoid", "target." + target)
    def sigmoid(expr):  # pylint: disable=unused-variable
        return True

    # 注册erf操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("erf", "target." + target)
    def erf(expr):  # pylint: disable=unused-variable
        return True

    def before():
        """创建包含条件分支的原始计算图

        返回:
            包含If-else结构的IRModule
        """
        # 创建输入变量
        data = relay.var("data", shape=(1, 32))
        eq1 = relay.var("e1", shape=[], dtype="float32")
        eq2 = relay.var("e2", shape=[], dtype="float32")
        # 创建条件表达式
        eq = relay.equal(eq1, eq2)

        # 创建条件为真时的分支（应用tanh激活函数）
        true_branch = relay.tanh(data)
        # 创建条件为假时的分支（应用sigmoid激活函数）
        false_branch = relay.sigmoid(data)
        # 创建条件分支结构
        ife = relay.If(eq, true_branch, false_branch)
        # 对条件分支的结果应用erf操作
        out = relay.erf(ife)
        func = relay.Function([data, eq1, eq2], out)
        mod = tvm.IRModule.from_expr(func)

        return mod

    def after():
        """生成预期的注解后计算图

        返回:
            包含正确编译器注解的条件分支IRModule
        """
        # 创建输入变量
        data = relay.var("data", shape=(1, 32))
        eq1 = relay.var("e1", shape=[], dtype="float32")
        eq2 = relay.var("e2", shape=[], dtype="float32")

        # 为条件变量添加编译器开始注解
        cb_1 = relay.annotation.compiler_begin(eq1, target)
        cb_2 = relay.annotation.compiler_begin(eq2, target)

        # 创建条件表达式并添加编译器结束注解
        equality_condition = relay.equal(cb_1, cb_2)
        ce_1 = relay.annotation.compiler_end(equality_condition, target)

        # 为条件为真的分支添加编译器注解
        cb_3 = relay.annotation.compiler_begin(data, target)
        true_branch = relay.tanh(cb_3)
        ce_2 = relay.annotation.compiler_end(true_branch, target)

        # 为条件为假的分支添加编译器注解
        cb_4 = relay.annotation.compiler_begin(data, target)
        false_branch = relay.sigmoid(cb_4)
        ce_3 = relay.annotation.compiler_end(false_branch, target)

        # 创建条件分支结构
        if_condition = relay.If(ce_1, ce_2, ce_3)
        # 为条件分支结果和最终erf操作添加编译器注解
        cb_5 = relay.annotation.compiler_begin(if_condition, target)
        erf_out = relay.erf(cb_5)
        ce_4 = relay.annotation.compiler_end(erf_out, target)
        func = relay.Function([data, eq1, eq2], ce_4)
        mod = tvm.IRModule.from_expr(func)
        return mod

    # 获取预期结果并应用类型推断
    expected = transform.InferType()(after())
    # 测试两种annotate_non_call_ops参数的情况
    for annotate_non_call_ops in [True, False]:
        # 应用AnnotateTarget转换
        result = transform.AnnotateTarget(target, annotate_non_call_ops)(before())
        # 验证实际结果与预期结果是否结构相等
        tvm.ir.assert_structural_equal(expected, result)


def test_while_let():
    """测试循环和Let绑定在AnnotateTarget转换中的处理

    此测试用例验证当计算图包含循环结构和Let绑定时，
    AnnotateTarget转换能否正确地处理这些复杂结构并添加编译器注解。
    """
    # 定义目标编译器标识
    target = "test_while_let"

    # 注册less操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("less", "target." + target)
    def less(expr):  # pylint: disable=unused-variable
        return True

    # 注册add操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("add", "target." + target)
    def add(expr):  # pylint: disable=unused-variable
        return True

    # 注册zeros_like操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("zeros_like", "target." + target)
    def zeros_like(expr):  # pylint: disable=unused-variable
        return True

    def before():
        """创建包含循环和Let绑定的原始计算图

        返回:
            包含循环和Let结构的IRModule
        """
        # 创建输入变量
        var1 = relay.var("var1", shape=(2,))
        var2 = relay.var("var2", shape=(), dtype="int32")
        var3 = relay.var("var3", shape=(2,))
        # 创建循环条件（var2 < 10）
        cond = relay.less(var2, relay.const(10, dtype="int32"))

        # 创建循环函数变量
        loop = relay.var("while_loop")
        # 循环计数器递增
        ii = var2 + relay.const(1, dtype="int32")
        # 累加计算
        ss = var3 + var1
        # 递归调用循环函数
        true_branch = loop(ii, ss)
        # 创建条件分支（循环继续或终止）
        ife = relay.If(cond, true_branch, var3)
        # 定义循环函数
        func_1 = relay.Function([var2, var3], ife)

        # 使用Let绑定定义循环函数并启动循环
        ret = relay.Let(loop, func_1, loop(relay.const(0, dtype="int32"), relay.zeros_like(var1)))
        # 创建主函数
        func_2 = relay.Function([var1], ret)
        mod = tvm.IRModule.from_expr(func_2)
        return mod

    def after(annotate_non_call_ops):
        """生成预期的注解后计算图

        参数:
            annotate_non_call_ops: 布尔值，表示是否为非调用操作添加注解

        返回:
            包含正确编译器注解的循环和Let结构IRModule
        """
        # 创建输入变量
        var1 = relay.var("var1", shape=(2,))
        var2 = relay.var("var2", shape=(), dtype="int32")
        var3 = relay.var("var3", shape=(2,))
        var4 = relay.const(10, dtype="int32")

        # 为循环条件变量添加编译器注解
        cb_1 = relay.annotation.compiler_begin(var2, target)
        cb_2 = relay.annotation.compiler_begin(var4, target)

        # 创建循环条件并添加编译器结束注解
        less_condition = relay.less(cb_1, cb_2)
        ce_1 = relay.annotation.compiler_end(less_condition, target)

        # 创建循环函数变量
        loop = relay.var("while_loop")

        # 为循环体内的操作添加编译器注解
        cb_3 = relay.annotation.compiler_begin(var2, target)
        cb_4 = relay.annotation.compiler_begin(relay.const(1, dtype="int32"), target)
        add_op_1 = relay.add(cb_3, cb_4)
        ce_2 = relay.annotation.compiler_end(add_op_1, target)

        # 根据参数决定是否为非调用操作添加注解
        cb_5 = relay.annotation.compiler_begin(ce_2, "default") if annotate_non_call_ops else ce_2

        # 为累加操作添加编译器注解
        cb_6 = relay.annotation.compiler_begin(var3, target)
        cb_7 = relay.annotation.compiler_begin(var1, target)
        add_op_2 = relay.add(cb_6, cb_7)
        ce_3 = relay.annotation.compiler_end(add_op_2, target)

        # 根据参数决定是否为非调用操作添加注解
        cb_8 = relay.annotation.compiler_begin(ce_3, "default") if annotate_non_call_ops else ce_3

        # 递归调用循环函数（循环体）
        true_branch = loop(cb_5, cb_8)  # while loop
        # 根据参数决定是否为非调用操作添加注解
        ce_4 = (
            relay.annotation.compiler_end(true_branch, "default")
            if annotate_non_call_ops
            else true_branch
        )
        # 创建条件分支
        if_condition = relay.If(ce_1, ce_4, var3)
        const_1 = relay.const(0, dtype="int32")
        # 根据参数决定是否为常量添加注解
        cb_9 = (
            relay.annotation.compiler_begin(const_1, "default")
            if annotate_non_call_ops
            else const_1
        )
        # 为zeros_like操作添加编译器注解
        cb_10 = relay.annotation.compiler_begin(var1, target)
        zeros_like = relay.zeros_like(cb_10)
        ce_5 = relay.annotation.compiler_end(zeros_like, target)
        # 根据参数决定是否为非调用操作添加注解
        cb_11 = relay.annotation.compiler_begin(ce_5, "default") if annotate_non_call_ops else ce_5
        # 启动循环
        while_condition = loop(cb_9, cb_11)
        # 根据参数决定是否为非调用操作添加注解
        ce_6 = (
            relay.annotation.compiler_end(while_condition, "default")
            if annotate_non_call_ops
            else while_condition
        )

        # 定义循环函数和主函数
        func_1 = relay.Function([var2, var3], if_condition)
        ret = relay.Let(loop, func_1, ce_6)
        func_2 = relay.Function([var1], ret)
        mod = tvm.IRModule.from_expr(func_2)
        return mod

    # 测试两种annotate_non_call_ops参数的情况
    for annotate_non_call_ops in [False, True]:
        # 应用AnnotateTarget转换
        result = transform.AnnotateTarget(target, annotate_non_call_ops)(before())
        # 获取预期结果并应用类型推断
        expected = transform.InferType()(after(annotate_non_call_ops))
        # 验证实际结果与预期结果是否结构相等
        tvm.ir.assert_structural_equal(expected, result)


def test_if_free_vars():
    """测试包含自由变量的条件分支在AnnotateTarget转换中的处理

    此测试用例验证当条件分支中包含自由变量（如常量生成操作）时，
    AnnotateTarget转换能否正确地处理这些结构并添加编译器注解。
    """
    # 定义目标编译器标识
    target = "test_if_free_vars"

    # 注册equal操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("equal", "target." + target)
    def equal(expr):  # pylint: disable=unused-variable
        return True

    # 注册sigmoid操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("sigmoid", "target." + target)
    def sigmoid(expr):  # pylint: disable=unused-variable
        return True

    # 注册erf操作在指定目标上的处理函数
    @tvm.ir.register_op_attr("erf", "target." + target)
    def erf(expr):  # pylint: disable=unused-variable
        return True

    def before():
        """创建包含自由变量的条件分支原始计算图

        返回:
            包含自由变量的条件分支IRModule
        """
        # 创建输入变量
        data = relay.var("data", shape=(1, 32))
        eq1 = relay.var("e1", shape=[], dtype="float32")
        eq2 = relay.var("e2", shape=[], dtype="float32")
        # 创建条件表达式
        eq = relay.equal(eq1, eq2)

        # 条件为真时返回零张量（自由变量示例）
        true_branch = relay.zeros(shape=(1, 32), dtype="float32")
        # 条件为假时应用sigmoid激活函数
        false_branch = relay.sigmoid(data)
        # 创建条件分支结构
        ife = relay.If(eq, true_branch, false_branch)
        # 对条件分支结果应用erf操作
        out = relay.erf(ife)

        func = relay.Function([data, eq1, eq2], out)
        mod = tvm.IRModule.from_expr(func)

        return mod

    def after():
        """生成预期的注解后计算图

        返回:
            包含正确编译器注解的自由变量条件分支IRModule
        """
        # 创建输入变量
        data = relay.var("data", shape=(1, 32))
        eq1 = relay.var("e1", shape=[], dtype="float32")
        eq2 = relay.var("e2", shape=[], dtype="float32")

        # 为条件变量添加编译器注解
        cb_1 = relay.annotation.compiler_begin(eq1, target)
        cb_2 = relay.annotation.compiler_begin(eq2, target)

        # 创建条件表达式并添加编译器结束注解
        equality_condition = relay.equal(cb_1, cb_2)
        ce_1 = relay.annotation.compiler_end(equality_condition, target)

        # 条件为真时返回零张量（自由变量，不需要添加注解）
        true_branch = relay.zeros(shape=(1, 32), dtype="float32")

        # 为条件为假的分支添加编译器注解
        cb_3 = relay.annotation.compiler_begin(data, target)
        false_branch = relay.sigmoid(cb_3)
        ce_2 = relay.annotation.compiler_end(false_branch, target)

        # 创建条件分支结构
        if_condition = relay.If(ce_1, true_branch, ce_2)
        # 为条件分支结果和最终erf操作添加编译器注解
        cb_4 = relay.annotation.compiler_begin(if_condition, target)
        erf_out = relay.erf(cb_4)
        ce_3 = relay.annotation.compiler_end(erf_out, target)
        func = relay.Function([data, eq1, eq2], ce_3)
        mod = tvm.IRModule.from_expr(func)
        return mod

    # 测试两种annotate_non_call_ops参数的情况
    for annotate_non_call_ops in [True, False]:
        # 应用AnnotateTarget转换
        result = transform.AnnotateTarget(target, annotate_non_call_ops)(before())
        # 获取预期结果并应用类型推断
        expected = transform.InferType()(after())
        # 验证实际结果与预期结果是否结构相等
        tvm.ir.assert_structural_equal(expected, result)


def test_free_vars_zeros():
    """测试自由变量在AnnotateTarget转换中的独立处理

    此测试用例验证自由变量（如零张量生成操作）在单独存在时，
    AnnotateTarget转换能否正确地处理它们。
    """
    # 定义目标编译器标识
    target = "test_free_vars_zeros"

    def before():
        """创建包含自由变量的原始计算图

        返回:
            仅包含零张量生成操作的IRModule
        """
        # 创建一个不接受参数，仅返回零张量的函数
        func = relay.Function([], relay.zeros(shape=(0), dtype="float32"))
        mod = tvm.IRModule.from_expr(func)
        return mod

    def after():
        """生成预期的注解后计算图

        返回:
            预期的不包含编译器注解的零张量IRModule
        """
        # 自由变量不应被添加编译器注解，因此预期结果与原始结果相同
        func = relay.Function([], relay.zeros(shape=(0), dtype="float32"))
        mod = tvm.IRModule.from_expr(func)
        return mod

    # 应用AnnotateTarget转换
    result = transform.AnnotateTarget(target)(before())
    # 获取预期结果并应用类型推断
    expected = transform.InferType()(after())
    # 验证实际结果与预期结果是否结构相等
    tvm.ir.assert_structural_equal(expected, result)


def test_empty_tuple():
    """测试空元组在AnnotateTarget转换中的处理

    此测试用例验证空元组是否能像无参数调用一样正确地被AnnotateTarget转换处理。
    """
    # 定义目标编译器标识
    target = "test_empty_tuple"

    def before():
        """创建包含空元组的原始计算图

        返回:
            仅包含空元组的IRModule
        """
        # 创建一个不接受参数，仅返回空元组的函数
        func = relay.Function([], relay.Tuple([]))
        mod = tvm.IRModule.from_expr(func)
        return mod

    def after():
        """生成预期的注解后计算图

        返回:
            预期的不包含编译器注解的空元组IRModule
        """
        # 空元组不应被添加编译器注解，因此预期结果与原始结果相同
        func = relay.Function([], relay.Tuple([]))
        mod = tvm.IRModule.from_expr(func)
        return mod

    # 测试两种annotate_non_call_ops参数的情况
    for annotate_non_call_ops in [True, False]:
        # 应用AnnotateTarget转换
        result = transform.AnnotateTarget(target, annotate_non_call_ops)(before())
        # 获取预期结果并应用类型推断
        expected = transform.InferType()(after())
        # 验证实际结果与预期结果是否结构相等
        tvm.ir.assert_structural_equal(expected, result)
